# PPO State-Selection Preflight

Visible, staged checks for the reconstructed research simulator. PPO is a fixed downstream validator, not the proposed algorithm. The held-out test cohort is not evaluated. Full scientific training remains disabled until the selected-state manifest, budget, and seeds are approved. Commands stream output live, emit idle heartbeats, and include at least the Last 200 lines in failure reports.

> Research-only simulation. Not a medical device and not for clinical dosing or patient care.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('Torch:', torch.__version__, flush=True)
print('CUDA runtime:', torch.version.cuda, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

repo = Path('/content/VitalDB-Feature-Selection')
remote = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
if not repo.exists():
    subprocess.run(['git', 'clone', remote, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo)
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()
from src.colab_workflow import run_streaming_command as run_command
head = run_command(['git', '-C', str(repo), 'rev-parse', 'HEAD'], stage='repository HEAD')
print('Repository commit:', head.stdout.strip(), flush=True)

In [ ]:
import pandas as pd
import torch

torch_before = torch.__version__
pandas_before = pd.__version__
dry_run = run_command(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '-r', 'requirements-rl.txt'],
    stage='RL dependency dry run',
)
if 'Would install torch-' in dry_run.stdout or 'Would install pandas-' in dry_run.stdout:
    raise RuntimeError('RL dependency profile would replace torch or pandas; aborting.')
run_command(
    [sys.executable, '-m', 'pip', 'install', 'stable-baselines3==2.9.0'],
    stage='RL dependency install',
)
import stable_baselines3
import pandas as pd_after
import torch as torch_after
if torch_after.__version__ != torch_before or pd_after.__version__ != pandas_before:
    raise RuntimeError('torch/pandas changed during RL dependency setup.')
if not torch_after.cuda.is_available():
    raise RuntimeError('CUDA disappeared after dependency installation.')
print('SB3:', stable_baselines3.__version__, flush=True)
print('Torch/CUDA preserved:', torch_after.__version__, torch_after.cuda.get_device_name(0), flush=True)
print('Pandas preserved:', pd_after.__version__, flush=True)

In [ ]:
import json

drive_project = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
dataset_dir = drive_project / 'data/modeling/full'
project_data_root = drive_project / 'data'
official_demographics_cache = project_data_root / 'clinical/vitaldb_ppo_cohort_demographics.csv'
protocol_dir = drive_project / 'outputs/ppo_protocol'
drive_output_root = drive_project / 'outputs/ppo_control_comparison'
required = {'train.npz', 'val.npz', 'dataset_metadata.json', 'preprocessing.pkl', 'splits'}
entries = {path.name for path in dataset_dir.iterdir()}
missing = sorted(required - entries)
print('Drive dataset:', dataset_dir, flush=True)
print('Dataset entries:', sorted(entries), flush=True)
if missing:
    raise FileNotFoundError(f'Modeling dataset is incomplete: {missing}')
run_command([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--dataset-dir', str(dataset_dir),
    '--project-data-root', str(project_data_root),
    '--allow-official-demographics-download',
    '--official-demographics-cache', str(official_demographics_cache),
    '--protocol-dir', str(protocol_dir),
    '--output-root', str(drive_output_root),
    '--initialize-only',
], stage='dataset and cohort preflight', cwd=repo)
protocol = json.loads((protocol_dir / 'frozen_ppo_protocol.json').read_text())
print('Legacy frozen-v1 protocol loaded for compatibility only:', protocol['protocol_hash'], flush=True)

In [ ]:
local_audit_root = Path('/content/vitaldb_rl_audits')
run_command([
    sys.executable, 'scripts/run_pkpd_simulator_validation.py',
    '--patient', 'middle_male',
    '--episode-duration-seconds', '120',
    '--output-dir', str(local_audit_root / 'pkpd'),
], stage='PK-PD simulator audit', cwd=repo)

In [ ]:
run_command([
    sys.executable, 'scripts/validate_rl_environment.py',
    '--duration-seconds', '120',
    '--output-dir', str(local_audit_root / 'environment'),
], stage='Gymnasium environment audit', cwd=repo)

In [ ]:
import shutil

local_smoke_root = Path('/content/ppo_primary_smoke')
drive_smoke_root = drive_project / 'outputs/ppo_primary_smoke'
run_command([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--smoke',
    '--smoke-timesteps', '1000',
    '--state-profile', 'original_reconstructed',
    '--seed', '42',
    '--device', 'cuda',
    '--smoke-output-dir', str(local_smoke_root),
], stage='1,000-step common-MLP PPO smoke', cwd=repo)
source = local_smoke_root / 'original_reconstructed/seed_42'
destination = drive_smoke_root / 'original_reconstructed/seed_42'
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, destination, dirs_exist_ok=True)
print('Smoke artifacts synced to:', destination, flush=True)

## Full scientific training gate

The old 20-run GRU/attention inventory is a legacy architecture experiment and is not the primary state-only comparison. This cell intentionally does not launch it.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    confirmation = input('Type I_UNDERSTAND_STATE_SELECTION_IS_PENDING: ').strip()
    if confirmation != 'I_UNDERSTAND_STATE_SELECTION_IS_PENDING':
        raise RuntimeError('Full training confirmation mismatch.')
    raise RuntimeError(
        'Primary full training remains locked until the selected manifest, budget, and seeds are approved.'
    )
print('Full training is disabled. Review smoke status and clipping diagnostics first.', flush=True)